# NULLXES GAG CAP VC — Colab CPU research run

Run in a hosted Google Colab CPU runtime only. This notebook clones the project from GitHub, mounts Drive with Google's authorization prompt, retrieves real provider data, computes baseline forecasts and risk, optionally extracts source-backed geopolitical reports with Qwen, saves artifacts, and opens the Streamlit research terminal through Colab's authenticated port proxy. No local runtime, fabricated observations, public tunnel, or brokerage integration.

In [ ]:
import os, sys, platform, json, subprocess, hashlib, re
from pathlib import Path
from datetime import date, timedelta, datetime, timezone
if 'COLAB_RELEASE_TAG' not in os.environ:
    raise RuntimeError('Hosted Google Colab is required; local runtimes are not supported.')
if Path('/dev/nvidia0').exists():
    raise RuntimeError('GPU detected. Select a CPU-only runtime before continuing.')
print(f'Hosted Colab CPU | Python {platform.python_version()} | cores={os.cpu_count()}')
RUN_STATUS = {}
def record_status(name, status, detail):
    RUN_STATUS[name] = {'status': status, 'detail': str(detail)}
    print(f'{name}: {status} — {detail}')
record_status('runtime', 'READY', 'Hosted Colab CPU confirmed; GPU disabled')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')  # Google requests explicit authorization
REPO_DIR = Path('/content/NULLXESGAG')
REPO_URL = 'https://github.com/MagistrTheOne/NULLXESGAG.git'
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True, text=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True, text=True)
if not (REPO_DIR / 'pyproject.toml').is_file():
    raise RuntimeError('GitHub checkout does not contain the expected GAG CAP VC project.')
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / 'src'))  # import the src-layout checkout without installing project extras
DATA_ROOT = Path('/content/drive/MyDrive/NULLXES/GAG-CAP-VC')
for folder in ('lake', 'forecasts', 'risk', 'geo', 'reports', 'models'):
    (DATA_ROOT / folder).mkdir(parents=True, exist_ok=True)
record_status('github_and_drive', 'READY', f'{REPO_URL} → {REPO_DIR}; results → {DATA_ROOT}')

In [ ]:
# Install only absent or below-minimum dependencies. Compatible preinstalled versions are left untouched.
import importlib.util
from importlib import metadata
DEPENDENCIES = [
    ('numpy', 'numpy', (1, 26)), ('pandas', 'pandas', (2, 1)),
    ('httpx', 'httpx', (0, 27)), ('pyarrow', 'pyarrow', (15, 0)),
    ('duckdb', 'duckdb', (1, 0)), ('scipy', 'scipy', (1, 12)),
    ('statsmodels', 'statsmodels', (0, 14)), ('arch', 'arch', (7, 0)),
    ('pydantic', 'pydantic', (2, 7)),
    ('exchange_calendars', 'exchange-calendars', (4, 5)),
    ('requests', 'requests', (2, 31)),
]
def version_tuple(text):
    parts = [int(x) for x in re.findall(r'\d+', text)[:3]]
    return tuple((parts + [0, 0, 0])[:3])
missing = []
for module_name, distribution, minimum in DEPENDENCIES:
    try:
        present = importlib.util.find_spec(module_name) is not None
        installed = metadata.version(distribution) if present else '0'
    except (ImportError, ValueError, metadata.PackageNotFoundError):
        present, installed = False, '0'
    if not present or version_tuple(installed) < minimum:
        missing.append(f'{distribution}>={minimum[0]}.{minimum[1]}')
if missing:
    install_env = os.environ.copy()
    if any(item.startswith('llama-cpp-python') for item in missing):
        install_env['CMAKE_ARGS'] = '-DGGML_CUDA=OFF'
        install_env['FORCE_CMAKE'] = '1'
    result = subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], env=install_env, capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError('Colab dependency installation failed: ' + result.stderr[-5000:])
    print('Installed missing/incompatible packages only:', missing)
else:
    print('All required packages are present at compatible versions; no installation needed.')
def ensure_optional(module_name, distribution, minimum):
    try:
        present = importlib.util.find_spec(module_name) is not None
        installed = metadata.version(distribution) if present else '0'
    except (ImportError, ValueError, metadata.PackageNotFoundError):
        present, installed = False, '0'
    if not present or version_tuple(installed) < minimum:
        if distribution == 'llama-cpp-python':
            os.environ['CMAKE_ARGS'] = '-DGGML_CUDA=OFF'
            os.environ['FORCE_CMAKE'] = '1'
        result = subprocess.run([sys.executable, '-m', 'pip', 'install', f'{distribution}>={minimum[0]}.{minimum[1]}'], capture_output=True, text=True)
        if result.returncode: return f'UNAVAILABLE: {result.stderr[-2000:]}'
    return 'AVAILABLE'
OPTIONAL_STATUS = {
    'streamlit': ensure_optional('streamlit', 'streamlit', (1, 39)),
    'huggingface_hub': ensure_optional('huggingface_hub', 'huggingface_hub', (0, 24)),
    'llama_cpp': ensure_optional('llama_cpp', 'llama-cpp-python', (0, 3)),
}
record_status('dependencies', 'READY', f'core checked {len(DEPENDENCIES)} packages; optional components: {OPTIONAL_STATUS}; installed only missing/incompatible packages: {missing}')

In [ ]:
import pandas as pd
import requests
from google.colab import userdata
from gagcap.ingestion.cbr import key_rate
from gagcap.ingestion.moex import candles
from gagcap.ingestion.fred import observations as fred_observations
from gagcap.storage.lake import ObservationLake
LAKE = ObservationLake(DATA_ROOT / 'lake')
OBSERVATION_FILE = DATA_ROOT / 'lake' / 'observations.parquet'
START_DATE = date(2018, 1, 1)
END_DATE = date.today() - timedelta(days=1)  # avoid a potentially incomplete current session
INGESTION = {}
SOURCE_URLS = {}
def ingest(name, fetch):
    try:
        rows = fetch()
        if not rows:
            raise RuntimeError('Provider returned an empty dataset')
        LAKE.write(rows)
        INGESTION[name] = {'status': 'AVAILABLE', 'rows': len(rows)}
        SOURCE_URLS[name] = sorted({row.source_url for row in rows})
        print(name, INGESTION[name])
    except Exception as exc:
        INGESTION[name] = {'status': 'DATA_UNAVAILABLE', 'reason': f'{type(exc).__name__}: {exc}'}
        print(name, INGESTION[name])
ingest('CBR_key_rate', lambda: key_rate(START_DATE, END_DATE))
ASSETS = [
    {'secid': 'SBER', 'engine': 'stock', 'market': 'shares', 'board': 'TQBR', 'unit': 'RUB_per_share'},
    {'secid': 'IMOEX', 'engine': 'stock', 'market': 'index', 'board': 'SNDX', 'unit': 'index_points'},
]
for asset in ASSETS:
    ingest('MOEX_' + asset['secid'], lambda asset=asset: candles(asset['secid'], START_DATE, END_DATE, engine=asset['engine'], market=asset['market'], board=asset['board'], unit=asset['unit']))
try:
    FRED_KEY = userdata.get('FRED_API_KEY')
except Exception:
    FRED_KEY = None
if FRED_KEY:
    os.environ['FRED_API_KEY'] = FRED_KEY
    ingest('FRED_CPIAUCSL', lambda: fred_observations('CPIAUCSL', START_DATE, END_DATE, unit='index_1982_84_100'))
else:
    INGESTION['FRED_CPIAUCSL'] = {'status': 'DATA_UNAVAILABLE', 'reason': 'FRED_API_KEY absent from Colab Secrets; continuing with CBR/MOEX.'}
    print('FRED_CPIAUCSL:', INGESTION['FRED_CPIAUCSL'])
(DATA_ROOT / 'reports' / 'ingestion_status.json').write_text(json.dumps(INGESTION, ensure_ascii=False, indent=2), encoding='utf-8')
record_status('real_data_ingestion', 'READY' if any(k.startswith('MOEX_') and v['status']=='AVAILABLE' for k,v in INGESTION.items()) else 'DATA_UNAVAILABLE', INGESTION)

In [ ]:
from gagcap.forecasting.backtesting import resolve_calendar_horizon
from gagcap.forecasting.baselines import random_walk
from gagcap.quant.models import Forecast
from gagcap.quant.returns import log_returns
from gagcap.quant.risk import historical_var_es, monte_carlo_normal
from gagcap.storage.forecast_store import write_forecasts
import exchange_calendars as xcals
FORECASTS = []
RISK_ROWS = []
ASSET_RESULTS = []
MACRO_ROWS = []
DATA_VERSION = hashlib.sha256(OBSERVATION_FILE.read_bytes()).hexdigest() if OBSERVATION_FILE.is_file() else 'DATA_UNAVAILABLE'
market_frame = LAKE.query("SELECT series_id, observation_date, value, unit, source_url FROM observations WHERE source='moex' AND value IS NOT NULL ORDER BY series_id, observation_date") if OBSERVATION_FILE.is_file() else pd.DataFrame()
macro_frame = LAKE.query("SELECT source, series_id, observation_date, value, unit, source_url FROM observations WHERE source IN ('cbr','fred') AND value IS NOT NULL ORDER BY observation_date") if OBSERVATION_FILE.is_file() else pd.DataFrame()
if not macro_frame.empty:
    for series_id, group in macro_frame.groupby('series_id'):
        latest = group.sort_values('observation_date').iloc[-1]
        MACRO_ROWS.append({'series_id': str(series_id), 'source': str(latest['source']), 'date': str(latest['observation_date']), 'value': float(latest['value']), 'unit': str(latest['unit']), 'source_url': str(latest['source_url'])})
try:
    calendar = xcals.get_calendar('XMOS')
except Exception as exc:
    calendar = None
    CALENDAR_ERROR = f'{type(exc).__name__}: {exc}'
else:
    CALENDAR_ERROR = None
for series_id, group in market_frame.groupby('series_id') if not market_frame.empty else []:
    group = group.sort_values('observation_date')
    idx = pd.DatetimeIndex(pd.to_datetime(group['observation_date'], utc=True))
    prices = pd.Series(group['value'].astype(float).to_numpy(), index=idx, name=str(series_id))
    unit = str(group.iloc[-1]['unit']); source_url = str(group.iloc[-1]['source_url'])
    try:
        if calendar is None: raise RuntimeError('XMOS exchange calendar unavailable: ' + str(CALENDAR_ERROR))
        if len(prices) < 21 or not prices.index.is_unique or not prices.index.is_monotonic_increasing:
            raise RuntimeError('Insufficient history or invalid session index')
        calendar_sessions = calendar.sessions_in_range(prices.index[0].date(), (prices.index[-1] + pd.Timedelta(days=100)).date())
        if calendar_sessions.tz is None: calendar_sessions = calendar_sessions.tz_localize('UTC')
        else: calendar_sessions = calendar_sessions.tz_convert('UTC')
        history_sessions = calendar_sessions[(calendar_sessions >= prices.index[0]) & (calendar_sessions <= prices.index[-1])]
        missing_sessions = history_sessions.difference(prices.index)
        outside_calendar = prices.index.difference(calendar_sessions)
        if len(missing_sessions) or len(outside_calendar):
            raise RuntimeError(f'Calendar/data mismatch: missing_sessions={len(missing_sessions)}, outside_calendar={len(outside_calendar)}')
        returns = log_returns(prices)
        risk = historical_var_es(returns, confidence=0.95)
        RISK_ROWS.append({'series_id': str(series_id), 'observation_date': str(prices.index[-1].date()), 'daily_historical_var_95_loss': risk.var_loss, 'daily_historical_es_95_loss': risk.expected_shortfall_loss, 'unit': risk.unit, 'method': risk.method, 'input_data_version': DATA_VERSION})
        origin = prices.index[-1]
        for days in (7, 30, 90):
            horizon_sessions, target_session = resolve_calendar_horizon(calendar_sessions, origin, days)
            simulation = monte_carlo_normal(mean_return=float(returns.mean()), volatility=float(returns.std(ddof=1)), horizon_sessions=horizon_sessions, simulations=20000, confidence=0.95, seed=20261017)
            point = float(random_walk(float(prices.iloc[-1]), horizon_sessions)[-1])
            lower = max(0.0, float(prices.iloc[-1]) * (1.0 + simulation['interval_lower']))
            upper = float(prices.iloc[-1]) * (1.0 + simulation['interval_upper'])
            forecast = Forecast('random_walk_baseline', origin.date(), target_session.date(), days, horizon_sessions, point, lower, upper, 0.95, unit, DATA_VERSION, 'Point estimate: last close carried forward. Interval: 95% IID normal historical log-return Monte Carlo; not calibrated.')
            FORECASTS.append(forecast)
            ASSET_RESULTS.append({'asset': str(series_id), 'unit': unit, 'as_of': str(origin.date()), 'latest_price': float(prices.iloc[-1]), 'source_url': source_url, 'horizon_calendar_days': days, 'target_session': str(target_session.date()), 'horizon_sessions': horizon_sessions, 'point_estimate': point, 'interval_lower': lower, 'interval_upper': upper, 'expected_return_interval_median': simulation['median_return'], 'interval_confidence': 0.95, 'model_id': 'random_walk_baseline + normal_iid_monte_carlo', 'assumptions': forecast.assumptions, 'input_data_version': DATA_VERSION})
    except Exception as exc:
        ASSET_RESULTS.append({'asset': str(series_id), 'status': 'FORECAST_UNAVAILABLE', 'reason': f'{type(exc).__name__}: {exc}', 'source_url': source_url})
        print(series_id, 'FORECAST_UNAVAILABLE:', type(exc).__name__, str(exc))
if FORECASTS:
    write_forecasts(FORECASTS, DATA_ROOT / 'forecasts' / 'forecasts.parquet')
if RISK_ROWS:
    pd.DataFrame(RISK_ROWS).to_parquet(DATA_ROOT / 'risk' / 'risk_metrics.parquet', index=False)
(DATA_ROOT / 'reports' / 'quant_results.json').write_text(json.dumps(ASSET_RESULTS, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
(DATA_ROOT / 'reports' / 'macro_factors.json').write_text(json.dumps(MACRO_ROWS, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
calendar_name = 'XMOS' if calendar is not None else CALENDAR_ERROR
record_status('quantitative_forecasts', 'AVAILABLE' if FORECASTS else 'DATA_UNAVAILABLE', f'forecasts={len(FORECASTS)}; risk rows={len(RISK_ROWS)}; calendar={calendar_name}')

In [ ]:
# Qwen downloads only to this Colab runtime. Headlines are retained as unverified reports unless source-backed extraction succeeds.
GEO_ROWS = []
QWEN_STATUS = 'QWEN_UNAVAILABLE'
try:
    if OPTIONAL_STATUS['huggingface_hub'] != 'AVAILABLE' or OPTIONAL_STATUS['llama_cpp'] != 'AVAILABLE': raise RuntimeError(f'Optional Qwen dependencies unavailable: {OPTIONAL_STATUS}')
    from huggingface_hub import HfApi, hf_hub_download
    gguf_files = [name for name in HfApi().list_repo_files('Qwen/Qwen3-0.6B-GGUF') if name.lower().endswith('.gguf')]
    if not gguf_files: raise RuntimeError('Official Qwen repository contains no GGUF files.')
    preferred = sorted(name for name in gguf_files if 'q4_k_m' in name.lower())
    MODEL_FILE = preferred[0] if preferred else sorted(gguf_files)[0]
    MODEL_PATH = hf_hub_download(repo_id='Qwen/Qwen3-0.6B-GGUF', filename=MODEL_FILE, local_dir='/content/nullxes-models')
    os.environ['QWEN_GGUF_PATH'] = MODEL_PATH
    from gagcap.intelligence.qwen import extract_event
    QWEN_STATUS = 'AVAILABLE'
except Exception as exc:
    QWEN_STATUS = f'QWEN_UNAVAILABLE: {type(exc).__name__}: {exc}'
print('Qwen:', QWEN_STATUS)
try:
    response = requests.get('https://api.gdeltproject.org/api/v2/doc/doc', params={'query':'(sanctions OR armed conflict OR export restrictions)', 'mode':'ArtList', 'format':'json', 'maxrecords':'8', 'timespan':'7d', 'sort':'HybridRel'}, timeout=30)
    response.raise_for_status()
    articles = response.json().get('articles', [])
    if not isinstance(articles, list): raise RuntimeError('GDELT response has no article list')
    for article in articles:
        title = str(article.get('title') or '').strip(); url = str(article.get('url') or '').strip()
        if not title or not url.startswith('https://'): continue
        item = {'title': title, 'source_url': url, 'domain': article.get('domain'), 'reported_at': article.get('seendate'), 'event_status': 'UNVERIFIED_REPORT', 'extraction_status': QWEN_STATUS}
        if QWEN_STATUS == 'AVAILABLE':
            try:
                extracted = extract_event(title, source_url=url, model_path=MODEL_PATH, cpu_threads=os.cpu_count(), context_size=4096)
                item['extraction_status'] = extracted.status
                if extracted.event is not None:
                    item.update({'event_type': extracted.event.event_type, 'countries': extracted.event.countries, 'sectors': extracted.event.sectors, 'assets': extracted.event.assets, 'evidence_quotes': extracted.event.evidence_quotes, 'event_status': 'UNVERIFIED_REPORT'})
                else:
                    item['extraction_reason'] = extracted.reason
            except Exception as exc:
                item['extraction_status'] = f'QWEN_UNAVAILABLE: {type(exc).__name__}: {exc}'
        GEO_ROWS.append(item)
except Exception as exc:
    GEO_STATUS = f'DATA_UNAVAILABLE: {type(exc).__name__}: {exc}'
else:
    GEO_STATUS = f'{len(GEO_ROWS)} source-linked reports; all remain unverified'
if GEO_ROWS:
    pd.DataFrame(GEO_ROWS).to_parquet(DATA_ROOT / 'geo' / 'geo_events.parquet', index=False)
record_status('geopolitical_sources_and_qwen', 'AVAILABLE' if GEO_ROWS else 'DATA_UNAVAILABLE', f'{GEO_STATUS}; Qwen={QWEN_STATUS}')

In [ ]:
def md_link(label, url):
    safe = str(url).replace(' ', '%20')
    return f'[{label}]({safe})' if safe.startswith('https://') else 'SOURCE_UNAVAILABLE'
lines = ['# NULLXES GAG CAP VC — Analytical report', '', f'Generated (UTC): {datetime.now(timezone.utc).isoformat()}', '', 'Research outputs only. Forecasts are experimental; no trading execution or performance claim.', '']
lines += ['## Asset forecasts', '']
if ASSET_RESULTS:
    for result in ASSET_RESULTS:
        if 'status' in result:
            lines.append(f"- **{result['asset']} — DATA_UNAVAILABLE:** {result['reason']}")
            continue
        lines.append(f"### {result['asset']}")
        lines.append(f"- As of {result['as_of']}: {result['latest_price']:,.4f} {result['unit']} ({md_link('source', result['source_url'])})")
        lines.append(f"- {result['horizon_calendar_days']} calendar days / {result['horizon_sessions']} exchange sessions; target {result['target_session']}")
        lines.append(f"- Point estimate: {result['point_estimate']:,.4f}; 95% interval: [{result['interval_lower']:,.4f}, {result['interval_upper']:,.4f}] {result['unit']}")
        lines.append(f"- Monte Carlo median cumulative return: {result['expected_return_interval_median']:.2%}; model: {result['model_id']}")
        lines.append(f"- Assumptions: {result['assumptions']}")
        risk = next((row for row in RISK_ROWS if row['series_id'] == result['asset']), None)
        lines.append(f"- Daily historical 95% VaR / ES: {risk['daily_historical_var_95_loss']:.2%} / {risk['daily_historical_es_95_loss']:.2%}" if risk else '- Risk: DATA_UNAVAILABLE')
        lines.append('')
else:
    lines.append('- DATA_UNAVAILABLE: no valid market series or calendar-matched forecast.')
lines += ['## Economic factors', '']
if MACRO_ROWS:
    for factor in MACRO_ROWS:
        lines.append(f"- {factor['series_id']} ({factor['source']}): {factor['value']} {factor['unit']} as of {factor['date']} — {md_link('source', factor['source_url'])}")
    lines.append('- Macro values are observed context only; this baseline does not claim a causal macro effect.')
else:
    lines.append('- DATA_UNAVAILABLE: no real CBR/FRED macro observation was retrieved.')
lines += ['', '## Geopolitical reports', '']
if GEO_ROWS:
    for event in GEO_ROWS:
        lines.append(f"- **UNVERIFIED_REPORT** ({event.get('reported_at') or 'publication time unavailable'}): {event['title']} — {md_link(event.get('domain') or 'source document', event['source_url'])}; extraction: {event.get('extraction_status')}")
        if event.get('evidence_quotes'): lines.append('  - Evidence: ' + ' / '.join(event['evidence_quotes']))
else:
    lines.append('- DATA_UNAVAILABLE: GDELT returned no source-linked reports.')
lines += ['', '## Sources and run status', '']
for source_name, urls in SOURCE_URLS.items():
    lines.append(f"- {source_name}: {', '.join(md_link(url, url) for url in urls)}")
lines += ['', '## Component availability', '']
for name, status in RUN_STATUS.items(): lines.append(f"- {name}: {status['status']} — {status['detail']}")
lines += ['', f'Dataset SHA-256: `{DATA_VERSION}`', '']
REPORT_TEXT = '\n'.join(lines)
REPORT_FILE = DATA_ROOT / 'reports' / 'analysis.md'
REPORT_FILE.write_text(REPORT_TEXT, encoding='utf-8')
if MACRO_ROWS: (DATA_ROOT / 'reports' / 'macro_factors.json').write_text(json.dumps(MACRO_ROWS, ensure_ascii=False, indent=2), encoding='utf-8')
if ASSET_RESULTS: (DATA_ROOT / 'reports' / 'quant_results.json').write_text(json.dumps(ASSET_RESULTS, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
print(REPORT_TEXT)
record_status('analytical_report', 'SAVED', REPORT_FILE)

In [ ]:
import time
os.environ['GAGCAP_LAKE_PATH'] = str(OBSERVATION_FILE)
os.environ['GAGCAP_REPORT_PATH'] = str(REPORT_FILE)
os.environ['GAGCAP_RISK_PATH'] = str(DATA_ROOT / 'risk' / 'risk_metrics.parquet')
os.environ['GAGCAP_GEO_PATH'] = str(DATA_ROOT / 'geo' / 'geo_events.parquet')
terminal_log = open(DATA_ROOT / 'reports' / 'streamlit.log', 'a', encoding='utf-8')
if OPTIONAL_STATUS['streamlit'] != 'AVAILABLE': raise RuntimeError('Streamlit terminal unavailable: ' + OPTIONAL_STATUS['streamlit'])
terminal = subprocess.Popen([sys.executable, '-m', 'streamlit', 'run', str(REPO_DIR / 'app.py'), '--server.port', '8501', '--server.address', '0.0.0.0', '--server.headless', 'true'], cwd=REPO_DIR, stdout=terminal_log, stderr=terminal_log)
from google.colab import output
display(output.serve_kernel_port_as_iframe(8501, height=900))
record_status('streamlit_terminal', 'RUNNING_IN_COLAB', 'Colab authenticated port proxy; no public tunnel or local server')